In [2]:
print("hi juhi")

hi juhi


In [5]:
# ============================================================
# BANK STRESS MONITOR
# Phase 3 — Sample Validation Notebook
# ============================================================
# Author:      Juhi Shah
# Date:        March 2026
# Purpose:     Validate historical data pull approach before
#              running full 2002-2023 dataset
#              Tests 3 sample quarters across different
#              regulatory regimes to identify field
#              definition changes and coverage issues
#
# Sample Quarters:
#   2005 Q4 (20051231) — Pre-crisis baseline
#   2009 Q4 (20091231) — Peak crisis period
#   2015 Q4 (20151231) — Post-Basel III adoption
#
# Key Questions:
#   1. Are all 20 fields consistently available?
#   2. Do any fields show definition changes over time?
#   3. Does default labeling logic work correctly?
#   4. Does trend (QoQ) calculation work correctly?
# ============================================================

import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sample quarters to test
SAMPLE_QUARTERS = ['20051231', '20091231', '20151231']

# Fields — same 20 as Phase 2
FIELDS = [
    "CERT", "REPDTE",
    "ASSET", "DEP", "DEPUNINS", "LNLSGR", "SC",
    "EQTOT", "INTAN", "CHBAL", "OTHBRF",
    "RBCT1J", "RBCT2",
    "LNATRES", "NAASSET", "P9ASSET", "ORE",
    "LNRECONS", "LNRENRES",
    "NETINC", "NIM",
]

# Ratio columns
RATIO_COLS = [
    'LEVERAGE_RATIO', 'TEXAS_RATIO', 'SECURITIES_RATIO',
    'UNINSURED_DEP_RATIO', 'ROA', 'LTD_RATIO', 'CRE_RATIO',
    'NIM_RATIO', 'CASH_RATIO', 'BORROW_RATIO'
]

# Stress direction
STRESS_DIRECTION = {
    'LEVERAGE_RATIO':      'LOW',
    'TEXAS_RATIO':         'HIGH',
    'SECURITIES_RATIO':    'HIGH',
    'UNINSURED_DEP_RATIO': 'HIGH',
    'ROA':                 'LOW',
    'LTD_RATIO':           'HIGH',
    'CRE_RATIO':           'HIGH',
    'NIM_RATIO':           'LOW',
    'CASH_RATIO':          'LOW',
    'BORROW_RATIO':        'HIGH',
}

print("Libraries imported successfully")
print(f"Sample quarters: {SAMPLE_QUARTERS}")
print(f"Fields to pull: {len(FIELDS) - 2} data fields + CERT + REPDTE")

Libraries imported successfully
Sample quarters: ['20051231', '20091231', '20151231']
Fields to pull: 19 data fields + CERT + REPDTE


In [10]:
# ============================================================
# CELL 2: PULL FDIC FAILURES LIST
# ============================================================
# Purpose: Get complete list of all US bank failures
#          This is our default label source
#          A bank-quarter observation is labeled default=1
#          if the bank failed within 2 years of that quarter
#
# Fields pulled:
#   CERT     — unique bank identifier (join key)
#   NAME     — bank name
#   FAILDATE — date of failure (YYYYMMDD format)
#   SAVR     — savings review type
#   RESTYPE  — resolution type (acquired, liquidated etc.)
# ============================================================

url_failures = "https://banks.data.fdic.gov/api/failures"

params_failures = {
    "fields": "CERT,NAME,FAILDATE,SAVR,RESTYPE",
    "limit": 10000,
    "offset": 0,
    "output": "json"
}

response_failures = requests.get(url_failures, params=params_failures)
data_failures = response_failures.json()
failures_df = pd.DataFrame(
    [item["data"] for item in data_failures["data"]]
)

# Convert FAILDATE to datetime
failures_df['FAILDATE'] = pd.to_datetime(
    failures_df['FAILDATE'], format='%Y-%m-%d', errors='coerce'
)

# Convert CERT to int for joining
failures_df['CERT'] = pd.to_numeric(
    failures_df['CERT'], errors='coerce'
).astype('Int64')

# Filter to failures within our study period (2003-2025)
failures_study = failures_df[
    failures_df['FAILDATE'] >= '2003-01-01'
].copy()

print(f"Total bank failures in FDIC database: {len(failures_df)}")
print(f"Failures from 2003 onwards: {len(failures_study)}")
print(f"\nFailures by year (2003-2015):")
failures_study['FAIL_YEAR'] = failures_study['FAILDATE'].dt.year
year_counts = failures_study['FAIL_YEAR'].value_counts().sort_index()
for year, count in year_counts.items():
    if year <= 2015:
        bar = '█' * (count // 5)
        print(f"  {year}: {count:>4} {bar}")

print(f"\nSample failures:")
print(failures_study[['NAME', 'FAILDATE', 'RESTYPE']].head(10).to_string())

Total bank failures in FDIC database: 4114
Failures from 2003 onwards: 0

Failures by year (2003-2015):

Sample failures:
Empty DataFrame
Columns: [NAME, FAILDATE, RESTYPE]
Index: []


In [12]:
# ============================================================
# CELL 2 FIX: Check FAILDATE format first
# ============================================================

# Re-pull and inspect raw format
response_failures = requests.get(url_failures, params=params_failures)
data_failures = response_failures.json()
failures_df = pd.DataFrame(
    [item["data"] for item in data_failures["data"]]
)

# Check raw format before converting
print("Raw FAILDATE sample:")
print(failures_df['FAILDATE'].head(10).tolist())
print(f"\nFAILDATE dtype: {failures_df['FAILDATE'].dtype}")
print(f"\nCERT sample:")
print(failures_df['CERT'].head(5).tolist())
print(f"\nTotal rows: {len(failures_df)}")
print(f"\nColumns: {list(failures_df.columns)}")

Raw FAILDATE sample:
['5/28/1934', '1/3/1935', '12/21/1936', '5/31/1985', '5/31/1985', '5/31/1985', '5/31/1985', '5/31/1985', '6/6/1985', '6/7/1985']

FAILDATE dtype: object

CERT sample:
[nan, nan, nan, 13797.0, 18388.0]

Total rows: 4114

Columns: ['FAILDATE', 'CERT', 'RESTYPE', 'SAVR', 'NAME', 'ID']


In [14]:
# ============================================================
# CELL 2: PULL FDIC FAILURES LIST (FIXED)
# ============================================================

url_failures = "https://banks.data.fdic.gov/api/failures"

params_failures = {
    "fields": "CERT,NAME,FAILDATE,SAVR,RESTYPE",
    "limit": 10000,
    "offset": 0,
    "output": "json"
}

response_failures = requests.get(url_failures, params=params_failures)
data_failures = response_failures.json()
failures_df = pd.DataFrame(
    [item["data"] for item in data_failures["data"]]
)

# Convert FAILDATE — format is M/D/YYYY
failures_df['FAILDATE'] = pd.to_datetime(
    failures_df['FAILDATE'], format='%m/%d/%Y', errors='coerce'
)

# Convert CERT to numeric
failures_df['CERT'] = pd.to_numeric(
    failures_df['CERT'], errors='coerce'
)

# Drop rows with no CERT — very old pre-FDIC failures
failures_df = failures_df.dropna(subset=['CERT'])
failures_df['CERT'] = failures_df['CERT'].astype(int)

# Filter to study period 2003 onwards
failures_study = failures_df[
    failures_df['FAILDATE'] >= '2003-01-01'
].copy()

# Add year column
failures_study['FAIL_YEAR'] = failures_study['FAILDATE'].dt.year

print(f"Total failures in FDIC database: {len(failures_df)}")
print(f"Failures from 2003 onwards:      {len(failures_study)}")
print(f"\nFailures by year:")
year_counts = failures_study['FAIL_YEAR'].value_counts().sort_index()
for year, count in year_counts.items():
    bar = '█' * (count // 5)
    print(f"  {year}: {count:>4}  {bar}")

print(f"\nSample recent failures:")
print(failures_study.nlargest(10, 'FAILDATE')[
    ['NAME', 'FAILDATE', 'RESTYPE']
].to_string())

Total failures in FDIC database: 3626
Failures from 2003 onwards:      569

Failures by year:
  2003:    3  
  2004:    4  
  2007:    3  
  2008:   30  ██████
  2009:  148  █████████████████████████████
  2010:  157  ███████████████████████████████
  2011:   92  ██████████████████
  2012:   51  ██████████
  2013:   24  ████
  2014:   18  ███
  2015:    8  █
  2016:    5  █
  2017:    8  █
  2019:    4  
  2020:    4  
  2023:    5  █
  2024:    2  
  2025:    2  
  2026:    1  

Sample recent failures:
                          NAME   FAILDATE  RESTYPE
3462  METROPOLITAN CAPITAL B&T 2026-01-30  FAILURE
3461  SANTA ANNA NATIONAL BANK 2025-06-27  FAILURE
3460      PULASKI SAVINGS BANK 2025-01-17  FAILURE
3459       FIRST NB OF LINDSAY 2024-10-18  FAILURE
3458             REPUBLIC BANK 2024-04-26  FAILURE
3456             CITIZENS BANK 2023-11-03  FAILURE
3455  HEARTLAND TRI-STATE BANK 2023-07-28  FAILURE
3454       FIRST REPUBLIC BANK 2023-05-01  FAILURE
3453            SIGNATURE BANK 2

In [16]:
# ============================================================
# CELL 3: PULL THREE SAMPLE QUARTERS
# ============================================================
# Purpose: Pull financial data for 3 specific quarters
#          to validate field availability and consistency
#          across different regulatory regimes
#
# 2005 Q4: Pre-crisis — baseline environment
# 2009 Q4: Peak crisis — maximum stress period
# 2015 Q4: Post-Basel III — new capital definitions
#
# Note: Pulling all banks per quarter regardless of
#       current active status — historical pull must
#       include banks that have since failed or merged
# ============================================================

url_fin = "https://banks.data.fdic.gov/api/financials"

sample_data = {}

for quarter in SAMPLE_QUARTERS:
    print(f"Pulling {quarter}...", end=' ')

    params = {
        "fields": ",".join(FIELDS),
        "filters": f"REPDTE:{quarter}",
        "limit": 10000,
        "offset": 0,
        "output": "json"
    }

    response = requests.get(url_fin, params=params)
    data = response.json()

    df_quarter = pd.DataFrame(
        [item["data"] for item in data["data"]]
    )

    # Convert numeric fields
    numeric_cols = [f for f in FIELDS
                    if f not in ['CERT', 'REPDTE']]
    for col in numeric_cols:
        df_quarter[col] = pd.to_numeric(
            df_quarter[col], errors='coerce'
        )

    # Convert CERT to int
    df_quarter['CERT'] = pd.to_numeric(
        df_quarter['CERT'], errors='coerce'
    ).astype('Int64')

    sample_data[quarter] = df_quarter
    print(f"{len(df_quarter):,} banks")

print(f"\nSample data pulled successfully")
print(f"\nBank counts by quarter:")
for quarter, df in sample_data.items():
    print(f"  {quarter}: {len(df):,} banks")

Pulling 20051231... 8,939 banks
Pulling 20091231... 8,104 banks
Pulling 20151231... 6,255 banks

Sample data pulled successfully

Bank counts by quarter:
  20051231: 8,939 banks
  20091231: 8,104 banks
  20151231: 6,255 banks


In [18]:
# ============================================================
# CELL 4: CROSS-PERIOD FIELD COVERAGE DIAGNOSTIC
# ============================================================
# Purpose: Compare field availability across 3 quarters
#          Identify fields with definition changes or
#          coverage gaps in earlier periods
#
# A field flagged here requires a decision:
#   - Consistently available → use as-is
#   - Poor early coverage → flag observations, don't impute
#   - Definition changed → add regime indicator variable
# ============================================================

# Numeric fields to check
check_fields = [f for f in FIELDS if f not in ['CERT', 'REPDTE']]

print("=" * 75)
print("CROSS-PERIOD FIELD COVERAGE DIAGNOSTIC")
print("Comparing field availability across regulatory regimes")
print("=" * 75)
print(f"\n{'Field':<15} {'2005 Q4':>12} {'2009 Q4':>12} "
      f"{'2015 Q4':>12} {'Consistent?':>12}")
print("-" * 75)

field_issues = []

for field in check_fields:
    coverage = []
    for quarter in SAMPLE_QUARTERS:
        df = sample_data[quarter]
        total = len(df)
        nulls = df[field].isna().sum()
        zeros = (df[field] == 0).sum()
        valid = total - nulls - zeros
        pct = 100 * valid / total
        coverage.append(pct)

    # Flag if coverage varies by more than 20% across periods
    max_diff = max(coverage) - min(coverage)
    consistent = '✅' if max_diff < 20 else '⚠️ CHECK'

    if max_diff >= 20:
        field_issues.append({
            'Field': field,
            '2005': coverage[0],
            '2009': coverage[1],
            '2015': coverage[2],
            'Max Diff': max_diff
        })

    print(f"{field:<15} {coverage[0]:>11.1f}% {coverage[1]:>11.1f}% "
          f"{coverage[2]:>11.1f}% {consistent:>12}")

print("\n" + "=" * 75)
print("FIELDS REQUIRING ATTENTION (coverage varies >20%):")
print("=" * 75)

if field_issues:
    for issue in sorted(field_issues,
                         key=lambda x: x['Max Diff'],
                         reverse=True):
        print(f"\n  {issue['Field']}:")
        print(f"    2005: {issue['2005']:.1f}% | "
              f"2009: {issue['2009']:.1f}% | "
              f"2015: {issue['2015']:.1f}%")
        print(f"    Max difference: {issue['Max Diff']:.1f}%")
else:
    print("  None — all fields consistent across periods ✅")

CROSS-PERIOD FIELD COVERAGE DIAGNOSTIC
Comparing field availability across regulatory regimes

Field                2005 Q4      2009 Q4      2015 Q4  Consistent?
---------------------------------------------------------------------------
ASSET                 100.0%       100.0%       100.0%            ✅
DEP                    99.0%        99.1%        99.1%            ✅
DEPUNINS               98.6%        97.7%        98.7%            ✅
LNLSGR                 98.2%        98.4%        98.4%            ✅
SC                     97.4%        96.7%        97.2%            ✅
EQTOT                  99.8%        99.8%        99.7%            ✅
INTAN                  37.4%        38.4%        44.4%            ✅
CHBAL                  99.9%        99.8%        99.9%            ✅
OTHBRF                 65.6%        66.9%        59.2%            ✅
RBCT1J                 99.8%        99.8%        99.7%            ✅
RBCT2                  97.7%        97.8%        97.9%            ✅
LNATRES      

In [20]:
# ============================================================
# CELL 4: CROSS-PERIOD FIELD COVERAGE DIAGNOSTIC
# ============================================================
# Purpose: Compare field availability across 4 quarters
#          including 2025 as the benchmark
#          Identify fields with definition changes or
#          coverage gaps in earlier periods
# ============================================================

# Add 2025 to comparison — load from Phase 2 output
# Re-pull 2025 Q4 for clean comparison
print("Pulling 2025 Q4 for benchmark comparison...", end=' ')
params_2025 = {
    "fields": ",".join(FIELDS),
    "filters": "REPDTE:20251231",
    "limit": 10000,
    "offset": 0,
    "output": "json"
}
response_2025 = requests.get(url_fin, params=params_2025)
data_2025 = response_2025.json()
df_2025 = pd.DataFrame(
    [item["data"] for item in data_2025["data"]]
)
numeric_cols = [f for f in FIELDS if f not in ['CERT', 'REPDTE']]
for col in numeric_cols:
    df_2025[col] = pd.to_numeric(df_2025[col], errors='coerce')
print(f"{len(df_2025):,} banks")

# Add to sample data dict
sample_data['20251231'] = df_2025

# All 4 quarters for comparison
all_quarters = ['20051231', '20091231', '20151231', '20251231']
quarter_labels = ['2005 Q4', '2009 Q4', '2015 Q4', '2025 Q4']

check_fields = [f for f in FIELDS if f not in ['CERT', 'REPDTE']]

print("\n" + "=" * 85)
print("CROSS-PERIOD FIELD COVERAGE DIAGNOSTIC")
print("Valid % per field per quarter (excludes nulls and zeros)")
print("=" * 85)
print(f"\n{'Field':<15} {'2005 Q4':>10} {'2009 Q4':>10} "
      f"{'2015 Q4':>10} {'2025 Q4':>10} {'Trend':>8} {'Flag':>8}")
print("-" * 85)

field_issues = []

for field in check_fields:
    coverage = []
    for quarter in all_quarters:
        df = sample_data[quarter]
        total = len(df)
        nulls = df[field].isna().sum()
        zeros = (df[field] == 0).sum()
        valid = total - nulls - zeros
        pct = 100 * valid / total
        coverage.append(pct)

    # Trend — improving, declining, or stable
    trend_val = coverage[3] - coverage[0]
    if trend_val > 10:
        trend = '↑ Rising'
    elif trend_val < -10:
        trend = '↓ Falling'
    else:
        trend = '→ Stable'

    # Flag if coverage varies by more than 20%
    max_diff = max(coverage) - min(coverage)
    flag = '⚠️' if max_diff >= 20 else '✅'

    if max_diff >= 20:
        field_issues.append({
            'Field': field,
            '2005': coverage[0],
            '2009': coverage[1],
            '2015': coverage[2],
            '2025': coverage[3],
            'Trend': trend_val,
            'Max Diff': max_diff
        })

    print(f"{field:<15} {coverage[0]:>9.1f}% {coverage[1]:>9.1f}% "
          f"{coverage[2]:>9.1f}% {coverage[3]:>9.1f}% "
          f"{trend:>10} {flag:>6}")

print("\n" + "=" * 85)
print("FIELDS REQUIRING ATTENTION (coverage varies > 20%):")
print("=" * 85)

if field_issues:
    issues_df = pd.DataFrame(field_issues).sort_values(
        'Max Diff', ascending=False
    )
    for _, issue in issues_df.iterrows():
        print(f"\n  {issue['Field']}:")
        print(f"    2005: {issue['2005']:.1f}% → "
              f"2009: {issue['2009']:.1f}% → "
              f"2015: {issue['2015']:.1f}% → "
              f"2025: {issue['2025']:.1f}%")
        print(f"    Max difference: {issue['Max Diff']:.1f}% | "
              f"Trend: {'+' if issue['Trend'] > 0 else ''}"
              f"{issue['Trend']:.1f}%")
else:
    print("  None — all fields consistent ✅")

print(f"\nBank counts by quarter:")
for q, label in zip(all_quarters, quarter_labels):
    print(f"  {label}: {len(sample_data[q]):,} banks")

Pulling 2025 Q4 for benchmark comparison... 4,408 banks

CROSS-PERIOD FIELD COVERAGE DIAGNOSTIC
Valid % per field per quarter (excludes nulls and zeros)

Field              2005 Q4    2009 Q4    2015 Q4    2025 Q4    Trend     Flag
-------------------------------------------------------------------------------------
ASSET               100.0%     100.0%     100.0%     100.0%   → Stable      ✅
DEP                  99.0%      99.1%      99.1%      98.7%   → Stable      ✅
DEPUNINS             98.6%      97.7%      98.7%      98.4%   → Stable      ✅
LNLSGR               98.2%      98.4%      98.4%      97.9%   → Stable      ✅
SC                   97.4%      96.7%      97.2%      97.5%   → Stable      ✅
EQTOT                99.8%      99.8%      99.7%      99.6%   → Stable      ✅
INTAN                37.4%      38.4%      44.4%      50.2%   ↑ Rising      ✅
CHBAL                99.9%      99.8%      99.9%      99.9%   → Stable      ✅
OTHBRF               65.6%      66.9%      59.2%      56.6

In [22]:
# ============================================================
# CELL 5: DEEP DIVE ON DEPUNINS ACROSS PERIODS
# ============================================================
# Purpose: Verify uninsured deposits field is genuinely
#          consistent — earlier analysis showed community
#          banks had poor coverage in 2025 with old field
#          name (DEPUNA). Confirming DEPUNINS is reliable
#          across all historical periods
# ============================================================

print("DEPUNINS COVERAGE BY SIZE TIER ACROSS PERIODS")
print("=" * 75)

# Need asset data to assign size tiers
print(f"\n{'Tier':<20} {'2005 Q4':>10} {'2009 Q4':>10} "
      f"{'2015 Q4':>10} {'2025 Q4':>10}")
print("-" * 75)

def get_tier(asset):
    if pd.isna(asset): return 'Unknown'
    elif asset < 1_000_000: return 'Community'
    elif asset < 10_000_000: return 'Mid-Size'
    elif asset < 100_000_000: return 'Large Regional'
    else: return 'Large'

for quarter in all_quarters:
    df = sample_data[quarter].copy()
    df['SIZE_TIER'] = df['ASSET'].apply(get_tier)
    sample_data[quarter] = df

tier_order = ['Community', 'Mid-Size', 'Large Regional', 'Large']

for tier in tier_order:
    row = f"{tier:<20}"
    for quarter in all_quarters:
        df = sample_data[quarter]
        tier_data = df[df['SIZE_TIER'] == tier]['DEPUNINS']
        total = len(tier_data)
        if total == 0:
            row += f"{'N/A':>10}"
            continue
        nulls = tier_data.isna().sum()
        zeros = (tier_data == 0).sum()
        valid = total - nulls - zeros
        pct = 100 * valid / total
        row += f"{pct:>9.1f}%"
    print(row)

print(f"\nNote: Valid % excludes both nulls AND zeros")
print(f"Zero DEPUNINS = bank reports zero uninsured deposits")
print(f"             = either genuinely zero OR not reporting")
print(f"             = indistinguishable from public data alone")

# Also check median value where valid
print(f"\nMedian DEPUNINS where valid (non-null, non-zero):")
print(f"{'Tier':<20} {'2005 Q4':>12} {'2009 Q4':>12} "
      f"{'2015 Q4':>12} {'2025 Q4':>12}")
print("-" * 75)

for tier in tier_order:
    row = f"{tier:<20}"
    for quarter in all_quarters:
        df = sample_data[quarter]
        tier_data = df[df['SIZE_TIER'] == tier]['DEPUNINS']
        valid_vals = tier_data.dropna()
        valid_vals = valid_vals[valid_vals != 0]
        if len(valid_vals) > 0:
            row += f"{valid_vals.median():>12,.0f}"
        else:
            row += f"{'N/A':>12}"
    print(row)

DEPUNINS COVERAGE BY SIZE TIER ACROSS PERIODS

Tier                    2005 Q4    2009 Q4    2015 Q4    2025 Q4
---------------------------------------------------------------------------
Community                98.6%     97.6%     98.6%     98.2%
Mid-Size                 98.7%     99.1%     99.0%     99.2%
Large Regional           99.1%     95.7%     97.6%     97.7%
Large                   100.0%    100.0%    100.0%    100.0%

Note: Valid % excludes both nulls AND zeros
Zero DEPUNINS = bank reports zero uninsured deposits
             = either genuinely zero OR not reporting
             = indistinguishable from public data alone

Median DEPUNINS where valid (non-null, non-zero):
Tier                      2005 Q4      2009 Q4      2015 Q4      2025 Q4
---------------------------------------------------------------------------
Community                 17,556      11,704      21,450      48,747
Mid-Size                 439,232     279,568     403,610     498,472
Large Regional        

In [24]:
# ============================================================
# CELL 6: CALCULATE RATIOS ON SAMPLE DATA
# ============================================================
# Purpose: Verify ratio calculations work correctly
#          across all historical periods
#          Uses identical logic to Phase 2
# ============================================================

def calculate_ratios(df):
    """
    Calculate all 10 stress ratios from raw FDIC fields
    Returns dataframe with ratio columns added
    Identical logic to Phase 2 — ensures consistency
    """
    d = df.copy()

    # Fill missing values conservatively
    for col in ['INTAN', 'RBCT2', 'P9ASSET', 'ORE', 'OTHBRF']:
        d[col] = d[col].fillna(0)

    # Intermediate calculations
    d['NPA']       = d['NAASSET'] + d['P9ASSET'] + d['ORE']
    d['TCE']       = d['EQTOT'] - d['INTAN']
    d['TOTAL_CAP'] = d['RBCT1J'] + d['RBCT2']
    d['CRE_LOANS'] = d['LNRECONS'] + d['LNRENRES']

    # 10 Stress Ratios
    d['LEVERAGE_RATIO']      = d['RBCT1J'] / d['ASSET']
    d['TEXAS_RATIO']         = d['NPA'] / (d['TCE'] + d['LNATRES'])
    d['SECURITIES_RATIO']    = d['SC'] / d['ASSET']
    d['UNINSURED_DEP_RATIO'] = d['DEPUNINS'] / d['DEP']
    d['ROA']                 = d['NETINC'] / d['ASSET']
    d['LTD_RATIO']           = d['LNLSGR'] / d['DEP']
    d['CRE_RATIO']           = d['CRE_LOANS'] / d['TOTAL_CAP']
    d['NIM_RATIO']           = d['NIM'] / d['ASSET']
    d['CASH_RATIO']          = d['CHBAL'] / d['ASSET']
    d['BORROW_RATIO']        = d['OTHBRF'] / d['ASSET']

    # Clean infinities
    for col in RATIO_COLS:
        d[col] = d[col].replace([np.inf, -np.inf], np.nan)

    # Flag negative TCE
    d['NEGATIVE_TCE'] = d['TCE'] < 0

    return d

# Apply to all sample quarters
print("Calculating ratios for all sample quarters...")
for quarter in all_quarters:
    sample_data[quarter] = calculate_ratios(sample_data[quarter])
    
    # Quick sanity check
    df = sample_data[quarter]
    valid_ratios = df[RATIO_COLS].notna().sum().sum()
    total_possible = len(df) * len(RATIO_COLS)
    completeness = 100 * valid_ratios / total_possible
    neg_tce = df['NEGATIVE_TCE'].sum()
    
    print(f"\n  {quarter}:")
    print(f"    Banks: {len(df):,}")
    print(f"    Overall ratio completeness: {completeness:.1f}%")
    print(f"    Negative TCE banks: {neg_tce}")
    
    # Show median of key ratios
    print(f"    Median ratios:")
    key_ratios = ['LEVERAGE_RATIO', 'TEXAS_RATIO', 
                  'ROA', 'CRE_RATIO']
    for ratio in key_ratios:
        median = df[ratio].median()
        print(f"      {ratio:<25}: {median:.4f}")

print("\nRatio calculation complete ✅")

Calculating ratios for all sample quarters...

  20051231:
    Banks: 8,939
    Overall ratio completeness: 99.7%
    Negative TCE banks: 1
    Median ratios:
      LEVERAGE_RATIO           : 0.0946
      TEXAS_RATIO              : 0.0328
      ROA                      : 0.0100
      CRE_RATIO                : 1.6290

  20091231:
    Banks: 8,104
    Overall ratio completeness: 99.7%
    Negative TCE banks: 19
    Median ratios:
      LEVERAGE_RATIO           : 0.0919
      TEXAS_RATIO              : 0.1449
      ROA                      : 0.0045
      CRE_RATIO                : 1.9057

  20151231:
    Banks: 6,255
    Overall ratio completeness: 99.7%
    Negative TCE banks: 0
    Median ratios:
      LEVERAGE_RATIO           : 0.1026
      TEXAS_RATIO              : 0.0583
      ROA                      : 0.0085
      CRE_RATIO                : 1.5473

  20251231:
    Banks: 4,408
    Overall ratio completeness: 99.6%
    Negative TCE banks: 2
    Median ratios:
      LEVERAGE_RATIO 

In [26]:
# ============================================================
# CELL 7: FULL RATIO COMPARISON ACROSS ALL PERIODS
# ============================================================
# Purpose: Compare median of all 10 ratios across 4 quarters
#          Validates economic intuition:
#          - Texas Ratio should spike in 2009
#          - ROA should drop in 2009
#          - CRE Ratio should be elevated in 2009
#          - Leverage should be lower in 2009 (capital erosion)
#          - ORE-driven metrics should peak in 2009
# ============================================================

print("=" * 85)
print("MEDIAN RATIO COMPARISON ACROSS PERIODS")
print("Economic validation — do ratios move as expected?")
print("=" * 85)

ratio_labels = {
    'LEVERAGE_RATIO':      'Leverage Ratio     [LOW=stress]',
    'TEXAS_RATIO':         'Texas Ratio        [HIGH=stress]',
    'SECURITIES_RATIO':    'Securities/Assets  [HIGH=stress]',
    'UNINSURED_DEP_RATIO': 'Uninsured Dep %    [HIGH=stress]',
    'ROA':                 'ROA                [LOW=stress]',
    'LTD_RATIO':           'LTD Ratio          [HIGH=stress]',
    'CRE_RATIO':           'CRE/Total Capital  [HIGH=stress]',
    'NIM_RATIO':           'NIM Ratio          [LOW=stress]',
    'CASH_RATIO':          'Cash/Assets        [LOW=stress]',
    'BORROW_RATIO':        'Borrowings/Assets  [HIGH=stress]',
}

print(f"\n{'Ratio':<35} {'2005 Q4':>10} {'2009 Q4':>10} "
      f"{'2015 Q4':>10} {'2025 Q4':>10} {'Expected?':>12}")
print("-" * 90)

for ratio in RATIO_COLS:
    medians = []
    for quarter in all_quarters:
        median = sample_data[quarter][ratio].median()
        medians.append(median)

    direction = STRESS_DIRECTION[ratio]

    # Check if 2009 moved in expected stress direction
    if direction == 'HIGH':
        # 2009 should be higher than 2005
        expected = medians[1] > medians[0]
        arrow = '↑' if medians[1] > medians[0] else '↓'
    else:
        # 2009 should be lower than 2005
        expected = medians[1] < medians[0]
        arrow = '↓' if medians[1] < medians[0] else '↑'

    status = f'✅ {arrow}' if expected else f'⚠️ {arrow}'
    label = ratio_labels[ratio]

    print(f"{label:<35} "
          f"{medians[0]:>10.4f} "
          f"{medians[1]:>10.4f} "
          f"{medians[2]:>10.4f} "
          f"{medians[3]:>10.4f} "
          f"{status:>12}")

print("\n" + "=" * 85)
print("ECONOMIC INTERPRETATION")
print("=" * 85)
print("""
Key observations to verify:
  ✅ Texas Ratio spikes in 2009 → asset quality deteriorated
  ✅ ROA drops in 2009 → banks became unprofitable
  ✅ Leverage drops in 2009 → capital erosion from losses
  ✅ NIM changes reflect rate environment shifts
  ✅ CRE ratio elevated 2009-2015 → concentration risk built up
""")

# Also show % change from 2005 to 2009 for each ratio
print(f"\n{'Ratio':<35} {'2005→2009 Change':>20} {'Direction':>12}")
print("-" * 70)
for ratio in RATIO_COLS:
    val_2005 = sample_data['20051231'][ratio].median()
    val_2009 = sample_data['20091231'][ratio].median()
    if val_2005 != 0:
        pct_change = 100 * (val_2009 - val_2005) / abs(val_2005)
        direction = STRESS_DIRECTION[ratio]
        if direction == 'HIGH':
            flag = '🔴 Stress↑' if pct_change > 0 else '🟢 Better'
        else:
            flag = '🔴 Stress↑' if pct_change < 0 else '🟢 Better'
        print(f"{ratio:<35} {pct_change:>+18.1f}%  {flag}")

MEDIAN RATIO COMPARISON ACROSS PERIODS
Economic validation — do ratios move as expected?

Ratio                                  2005 Q4    2009 Q4    2015 Q4    2025 Q4    Expected?
------------------------------------------------------------------------------------------
Leverage Ratio     [LOW=stress]         0.0946     0.0919     0.1026     0.1077          ✅ ↓
Texas Ratio        [HIGH=stress]        0.0328     0.1449     0.0583     0.0295          ✅ ↑
Securities/Assets  [HIGH=stress]        0.1910     0.1659     0.1887     0.1781         ⚠️ ↓
Uninsured Dep %    [HIGH=stress]        0.2067     0.1075     0.1634     0.2451         ⚠️ ↓
ROA                [LOW=stress]         0.0100     0.0045     0.0085     0.0107          ✅ ↓
LTD Ratio          [HIGH=stress]        0.8217     0.8141     0.7874     0.8116         ⚠️ ↓
CRE/Total Capital  [HIGH=stress]        1.6290     1.9057     1.5473     1.7340          ✅ ↑
NIM Ratio          [LOW=stress]         0.0359     0.0332     0.0324     0.

In [28]:
# ============================================================
# CELL 8: TIER-SEPARATED RATIO COMPARISON ACROSS PERIODS
# ============================================================
# Purpose: Compare median ratios by size tier across periods
#          Addresses two issues:
#          1. Different bank counts per period distort medians
#          2. Community banks dominate and skew all-bank stats
#          Tier separation ensures apples-to-apples comparison
# ============================================================

tier_order = ['Community', 'Mid-Size', 'Large Regional', 'Large']

for tier in tier_order:
    print("=" * 90)
    print(f"SIZE TIER: {tier}")
    print("=" * 90)

    # Bank counts per period for this tier
    counts = []
    for quarter in all_quarters:
        df = sample_data[quarter]
        count = (df['SIZE_TIER'] == tier).sum()
        counts.append(count)

    print(f"Bank counts: 2005={counts[0]:,} | 2009={counts[1]:,} | "
          f"2015={counts[2]:,} | 2025={counts[3]:,}")

    print(f"\n{'Ratio':<35} {'2005 Q4':>10} {'2009 Q4':>10} "
          f"{'2015 Q4':>10} {'2025 Q4':>10} {'2005→2009':>12} {'Expected?':>10}")
    print("-" * 95)

    for ratio in RATIO_COLS:
        medians = []
        for quarter in all_quarters:
            df = sample_data[quarter]
            tier_data = df[df['SIZE_TIER'] == tier][ratio]
            medians.append(tier_data.median())

        direction = STRESS_DIRECTION[ratio]

        # % change 2005 to 2009
        if medians[0] and medians[0] != 0:
            pct_change = 100 * (medians[1] - medians[0]) / abs(medians[0])
            change_str = f"{pct_change:+.1f}%"
        else:
            change_str = "N/A"

        # Expected direction check
        if direction == 'HIGH':
            expected = medians[1] > medians[0]
        else:
            expected = medians[1] < medians[0]
        status = '✅' if expected else '⚠️'

        print(f"{ratio:<35} "
              f"{medians[0]:>10.4f} "
              f"{medians[1]:>10.4f} "
              f"{medians[2]:>10.4f} "
              f"{medians[3]:>10.4f} "
              f"{change_str:>12} "
              f"{status:>10}")

    print()

SIZE TIER: Community
Bank counts: 2005=8,301 | 2009=7,420 | 2015=5,540 | 2025=3,338

Ratio                                  2005 Q4    2009 Q4    2015 Q4    2025 Q4    2005→2009  Expected?
-----------------------------------------------------------------------------------------------
LEVERAGE_RATIO                          0.0960     0.0931     0.1036     0.1100        -3.1%          ✅
TEXAS_RATIO                             0.0322     0.1379     0.0573     0.0240      +328.4%          ✅
SECURITIES_RATIO                        0.1921     0.1659     0.1908     0.1895       -13.6%         ⚠️
UNINSURED_DEP_RATIO                     0.2000     0.1027     0.1556     0.2247       -48.7%         ⚠️
ROA                                     0.0099     0.0046     0.0083     0.0106       -53.5%          ✅
LTD_RATIO                               0.8129     0.8074     0.7726     0.7859        -0.7%         ⚠️
CRE_RATIO                               1.5785     1.8299     1.4302     1.4572       +15.9

In [30]:
# ============================================================
# CELL 9: TEST DEFAULT LABELING LOGIC
# ============================================================
# Purpose: Verify default flag assignment works correctly
#          Rules:
#          1. Default = 1 if bank failed within 2 years
#             of the observation quarter end date
#          2. Exclude observations AFTER failure date
#             (post-failure data is not predictive)
#          3. Default = 0 for all other observations
#
# Test on 2009 Q4 — should capture banks that failed
# between 2009-12-31 and 2011-12-31
# ============================================================

def assign_default_labels(df, quarter_date, failures_df,
                           window_years=2):
    """
    Assign default labels to bank-quarter observations

    Parameters:
    -----------
    df           : dataframe of bank observations for one quarter
    quarter_date : end date of the quarter (datetime)
    failures_df  : FDIC failures dataframe
    window_years : years forward to look for failures (default 2)

    Returns:
    --------
    dataframe with DEFAULT flag and exclusion mask added
    """
    d = df.copy()

    # Define the window end date
    window_end = quarter_date + pd.DateOffset(years=window_years)

    # For each bank, check if it failed within the window
    # Merge failure dates onto observation data
    d = d.merge(
        failures_df[['CERT', 'FAILDATE']],
        on='CERT',
        how='left'
    )

    # Default = 1 if failed within [quarter_date, window_end]
    d['DEFAULT'] = (
        (d['FAILDATE'] >= quarter_date) &
        (d['FAILDATE'] <= window_end)
    ).astype(int)

    # Exclude flag = True if observation is AFTER failure
    # These observations should be removed from training data
    d['POST_FAILURE'] = (
        d['FAILDATE'] < quarter_date
    ).fillna(False)

    # Clean up — drop FAILDATE (keep only flags)
    d = d.drop(columns=['FAILDATE'])

    return d

# Test on 2009 Q4
quarter_date = pd.Timestamp('2009-12-31')
df_2009 = sample_data['20091231'].copy()
df_2009_labeled = assign_default_labels(
    df_2009, quarter_date, failures_df
)

# Results
total = len(df_2009_labeled)
defaults = df_2009_labeled['DEFAULT'].sum()
post_failure = df_2009_labeled['POST_FAILURE'].sum()
clean = total - post_failure

print("=" * 60)
print("DEFAULT LABELING TEST — 2009 Q4")
print("=" * 60)
print(f"\nTotal observations:        {total:,}")
print(f"Post-failure (exclude):    {post_failure:,}")
print(f"Clean observations:        {clean:,}")
print(f"Default = 1 (failed 2yr):  {defaults:,}")
print(f"Default = 0 (healthy):     {clean - defaults:,}")
print(f"Default rate:              {100*defaults/clean:.2f}%")

print(f"\nSample DEFAULT=1 banks (failed 2010-2011):")
defaulted = df_2009_labeled[
    df_2009_labeled['DEFAULT'] == 1
][['CERT', 'NAME', 'SIZE_TIER', 'ASSET',
   'TEXAS_RATIO', 'ROA', 'LEVERAGE_RATIO']].head(10)

for _, row in defaulted.iterrows():
    print(f"  {str(row['NAME']):<40} "
          f"Texas: {row['TEXAS_RATIO']:.3f} | "
          f"ROA: {row['ROA']:.4f} | "
          f"Leverage: {row['LEVERAGE_RATIO']:.3f}")

print(f"\nSample POST_FAILURE banks (already failed before 2009):")
post = df_2009_labeled[
    df_2009_labeled['POST_FAILURE'] == True
][['CERT', 'NAME', 'SIZE_TIER']].head(5)
print(post.to_string())

DEFAULT LABELING TEST — 2009 Q4

Total observations:        8,105
Post-failure (exclude):    18
Clean observations:        8,087
Default = 1 (failed 2yr):  249
Default = 0 (healthy):     7,838
Default rate:              3.08%

Sample DEFAULT=1 banks (failed 2010-2011):


KeyError: "['NAME'] not in index"

In [32]:
# ============================================================
# CELL 9 FIX: Show defaulted banks without NAME column
# ============================================================

print("=" * 60)
print("DEFAULT LABELING TEST — 2009 Q4")
print("=" * 60)
print(f"\nTotal observations:        {total:,}")
print(f"Post-failure (exclude):    {post_failure:,}")
print(f"Clean observations:        {clean:,}")
print(f"Default = 1 (failed 2yr):  {defaults:,}")
print(f"Default = 0 (healthy):     {clean - defaults:,}")
print(f"Default rate:              {100*defaults/clean:.2f}%")

# Show defaulted banks — join with failures_df to get names
print(f"\nSample DEFAULT=1 banks (failed within 2 years of 2009 Q4):")
defaulted = df_2009_labeled[
    df_2009_labeled['DEFAULT'] == 1
][['CERT', 'SIZE_TIER', 'ASSET',
   'TEXAS_RATIO', 'ROA', 'LEVERAGE_RATIO']].head(10)

# Add names from failures_df
defaulted = defaulted.merge(
    failures_df[['CERT', 'NAME', 'FAILDATE']],
    on='CERT', how='left'
)

for _, row in defaulted.iterrows():
    print(f"  {str(row['NAME']):<40} "
          f"Failed: {str(row['FAILDATE'])[:10]} | "
          f"Texas: {row['TEXAS_RATIO']:.3f} | "
          f"ROA: {row['ROA']:.4f} | "
          f"Leverage: {row['LEVERAGE_RATIO']:.3f}")

print(f"\nPost-failure exclusions (already failed before 2009 Q4):")
post = df_2009_labeled[
    df_2009_labeled['POST_FAILURE'] == True
][['CERT', 'SIZE_TIER', 'ASSET']].head(5)

post = post.merge(
    failures_df[['CERT', 'NAME', 'FAILDATE']],
    on='CERT', how='left'
)
print(post[['NAME', 'FAILDATE', 'SIZE_TIER']].to_string())

DEFAULT LABELING TEST — 2009 Q4

Total observations:        8,105
Post-failure (exclude):    18
Clean observations:        8,087
Default = 1 (failed 2yr):  249
Default = 0 (healthy):     7,838
Default rate:              3.08%

Sample DEFAULT=1 banks (failed within 2 years of 2009 Q4):
  BANK OF HIAWASSEE                        Failed: 2010-03-19 | Texas: 4.050 | ROA: -0.0812 | Leverage: 0.014
  WESTERN SPRINGS NATIONAL BANK AND TRUST  Failed: 2011-04-08 | Texas: 1.646 | ROA: -0.0132 | Leverage: 0.067
  THE RIVERBANK                            Failed: 2011-10-07 | Texas: 1.727 | ROA: -0.0413 | Leverage: 0.059
  THUNDER BANK                             Failed: 2010-07-23 | Texas: 0.409 | ROA: -0.0183 | Leverage: 0.060
  PEOTONE BANK AND TRUST COMPANY           Failed: 2010-04-23 | Texas: 3.598 | ROA: -0.0742 | Leverage: 0.005
  HOME NATIONAL BANK                       Failed: 2010-07-09 | Texas: 2.453 | ROA: -0.0946 | Leverage: 0.018
  FIRST COMMUNITY BANK                     Failed: 201

In [34]:
# ============================================================
# CELL 10: TEST TREND (QoQ) CALCULATION LOGIC
# ============================================================
# Purpose: Verify quarter-over-quarter change calculation
#          works correctly for all 10 ratios
#
# Logic:
#   Trend = Current Quarter Ratio - Previous Quarter Ratio
#   Raw difference — not percentage change
#   Requires previous quarter data for each bank
#
# Test: Calculate trends between 2009 Q3 and 2009 Q4
#       Pull 2009 Q3 as the "previous quarter"
#       Merge with 2009 Q4 on CERT
#       Calculate difference for all 10 ratios
#
# Edge cases to handle:
#   - Bank exists in Q4 but not Q3 → trend = NaN
#   - Bank exists in Q3 but not Q4 → not in dataset
#   - Bank just chartered → first quarter, no trend
# ============================================================

# Pull previous quarter — 2009 Q3
print("Pulling 2009 Q3 for trend calculation test...", end=' ')
params_q3 = {
    "fields": ",".join(FIELDS),
    "filters": "REPDTE:20090930",
    "limit": 10000,
    "offset": 0,
    "output": "json"
}
response_q3 = requests.get(url_fin, params=params_q3)
data_q3 = response_q3.json()
df_q3 = pd.DataFrame(
    [item["data"] for item in data_q3["data"]]
)

# Convert numeric
for col in [f for f in FIELDS if f not in ['CERT', 'REPDTE']]:
    df_q3[col] = pd.to_numeric(df_q3[col], errors='coerce')
df_q3['CERT'] = pd.to_numeric(
    df_q3['CERT'], errors='coerce'
).astype('Int64')

# Calculate ratios for Q3
df_q3 = calculate_ratios(df_q3)
print(f"{len(df_q3):,} banks")

# Merge Q4 and Q3 on CERT
df_q4 = sample_data['20091231'].copy()

# Add _Q3 suffix to ratio columns for Q3
ratio_cols_q3 = {col: f"{col}_Q3" for col in RATIO_COLS}
df_q3_ratios = df_q3[['CERT'] + RATIO_COLS].rename(
    columns=ratio_cols_q3
)

# Merge
df_trend = df_q4.merge(df_q3_ratios, on='CERT', how='left')

# Calculate QoQ trend for all 10 ratios
trend_cols = []
for col in RATIO_COLS:
    trend_col = f"{col}_TREND"
    df_trend[trend_col] = df_trend[col] - df_trend[f"{col}_Q3"]
    trend_cols.append(trend_col)

# Summary
print("\n" + "=" * 65)
print("TREND CALCULATION TEST — 2009 Q3 → Q4")
print("=" * 65)

total_banks = len(df_trend)
banks_with_trends = df_trend[trend_cols].notna().any(axis=1).sum()
banks_no_prior = total_banks - banks_with_trends

print(f"\nTotal banks in Q4:           {total_banks:,}")
print(f"Banks with Q3 data (trends): {banks_with_trends:,}")
print(f"Banks without Q3 (new):      {banks_no_prior:,}")
print(f"Trend coverage:              "
      f"{100*banks_with_trends/total_banks:.1f}%")

# Show distribution of trend values
print(f"\nTrend distributions (median QoQ change):")
print(f"{'Ratio':<30} {'Median Trend':>15} "
      f"{'P10':>10} {'P90':>10}")
print("-" * 70)

for col, trend_col in zip(RATIO_COLS, trend_cols):
    valid = df_trend[trend_col].dropna()
    if len(valid) > 0:
        print(f"{col:<30} {valid.median():>+15.5f} "
              f"{valid.quantile(0.10):>+10.5f} "
              f"{valid.quantile(0.90):>+10.5f}")

# Show sample banks with large trend changes
print(f"\nTop 5 banks with largest TEXAS_RATIO deterioration Q3→Q4:")
worst = df_trend.nlargest(5, 'TEXAS_RATIO_TREND')[
    ['CERT', 'SIZE_TIER', 'TEXAS_RATIO',
     'TEXAS_RATIO_Q3', 'TEXAS_RATIO_TREND']
]
worst = worst.merge(
    failures_df[['CERT', 'NAME']], on='CERT', how='left'
)
for _, row in worst.iterrows():
    print(f"  {str(row['NAME']):<40} "
          f"Q3: {row['TEXAS_RATIO_Q3']:.3f} → "
          f"Q4: {row['TEXAS_RATIO']:.3f} "
          f"(Δ {row['TEXAS_RATIO_TREND']:+.3f})")

Pulling 2009 Q3 for trend calculation test... 8,191 banks

TREND CALCULATION TEST — 2009 Q3 → Q4

Total banks in Q4:           8,104
Banks with Q3 data (trends): 8,100
Banks without Q3 (new):      4
Trend coverage:              100.0%

Trend distributions (median QoQ change):
Ratio                             Median Trend        P10        P90
----------------------------------------------------------------------
LEVERAGE_RATIO                        -0.00144   -0.01269   +0.00478
TEXAS_RATIO                           +0.00049   -0.05167   +0.11991
SECURITIES_RATIO                      -0.00160   -0.02854   +0.02532
UNINSURED_DEP_RATIO                   +0.00111   -0.03363   +0.04415
ROA                                   +0.00091   -0.00867   +0.00359
LTD_RATIO                             -0.01473   -0.07594   +0.03861
CRE_RATIO                             +0.00000   -0.18647   +0.26276
NIM_RATIO                             +0.00820   +0.00533   +0.01091
CASH_RATIO                     

In [36]:
# ============================================================
# MEMORY AND VOLUME ESTIMATION
# ============================================================
# Purpose: Estimate full historical dataset size before
#          running the full pull
# ============================================================

# Quarters from 2002 Q4 to 2023 Q4
# 2002 Q4 = baseline for trend calculation
# 2003 Q1 to 2023 Q4 = training data = 84 quarters
# Plus 2002 Q4 baseline = 85 total pulls

quarters_to_pull = 85

# Average banks per quarter (declining from ~8900 to ~4400)
avg_banks_per_quarter = (8939 + 4408) / 2
estimated_rows = quarters_to_pull * avg_banks_per_quarter

# Columns in final dataset
# 20 raw fields + 10 ratios + 10 trends + ~10 metadata = ~50 cols
estimated_cols = 50

# Memory estimate
# Each cell ≈ 8 bytes (float64)
memory_bytes = estimated_rows * estimated_cols * 8
memory_mb = memory_bytes / (1024 * 1024)
memory_gb = memory_mb / 1024

# File size estimate (CSV is roughly 2x memory)
file_size_mb = memory_mb * 2

print("=" * 55)
print("FULL HISTORICAL DATASET — SIZE ESTIMATION")
print("=" * 55)
print(f"\nQuarters to pull:        {quarters_to_pull}")
print(f"Avg banks per quarter:   {avg_banks_per_quarter:,.0f}")
print(f"Estimated total rows:    {estimated_rows:,.0f}")
print(f"Estimated columns:       {estimated_cols}")
print(f"\nMemory estimate:         {memory_mb:,.0f} MB "
      f"({memory_gb:.2f} GB)")
print(f"CSV file size estimate:  {file_size_mb:,.0f} MB")
print(f"\nAPI calls required:      {quarters_to_pull}")
print(f"Est. time @ 20s/call:    "
      f"{quarters_to_pull * 20 / 60:.0f} minutes")
print(f"Est. time @ 30s/call:    "
      f"{quarters_to_pull * 30 / 60:.0f} minutes")

# Check available memory
import psutil
available_ram = psutil.virtual_memory().available / (1024**3)
total_ram = psutil.virtual_memory().total / (1024**3)
print(f"\nYour machine:")
print(f"  Total RAM:     {total_ram:.1f} GB")
print(f"  Available RAM: {available_ram:.1f} GB")
print(f"\nMemory sufficient: "
      f"{'✅ Yes' if available_ram > memory_gb * 2 else '⚠️ Tight — consider chunked processing'}")

FULL HISTORICAL DATASET — SIZE ESTIMATION

Quarters to pull:        85
Avg banks per quarter:   6,674
Estimated total rows:    567,248
Estimated columns:       50

Memory estimate:         216 MB (0.21 GB)
CSV file size estimate:  433 MB

API calls required:      85
Est. time @ 20s/call:    28 minutes
Est. time @ 30s/call:    42 minutes

Your machine:
  Total RAM:     8.0 GB
  Available RAM: 1.4 GB

Memory sufficient: ✅ Yes


In [38]:
# ============================================================
# CELL 11: QUARTER LOOP TEST — 4 CONSECUTIVE QUARTERS
# ============================================================
# Purpose: Test the full pipeline loop on 4 quarters
#          2008 Q1 → Q4 before running full 85-quarter pull
#
# Tests:
#   1. API loop pulls data correctly per quarter
#   2. Data appends correctly across quarters
#   3. Trend calculation works across consecutive quarters
#   4. Default labeling works in loop context
#   5. No memory or data corruption issues
#
# Quarter dates for 2008:
#   Q1: 20080331
#   Q2: 20080630
#   Q3: 20080930
#   Q4: 20081231
# ============================================================

import time

# Generate quarter end dates
def generate_quarter_dates(start_year, start_q, end_year, end_q):
    """Generate list of quarter end dates in YYYYMMDD format"""
    quarter_ends = {1: '0331', 2: '0630', 3: '0930', 4: '1231'}
    dates = []
    year, q = start_year, start_q
    while (year < end_year) or (year == end_year and q <= end_q):
        dates.append(f"{year}{quarter_ends[q]}")
        q += 1
        if q > 4:
            q = 1
            year += 1
    return dates

# Test quarters — 2007 Q4 as baseline + 2008 Q1-Q4
test_quarters = generate_quarter_dates(2007, 4, 2008, 4)
print(f"Test quarters: {test_quarters}")

def pull_quarter(quarter_date, fields):
    """Pull financial data for one quarter from FDIC API"""
    params = {
        "fields": ",".join(fields),
        "filters": f"REPDTE:{quarter_date}",
        "limit": 10000,
        "offset": 0,
        "output": "json"
    }
    response = requests.get(
        "https://banks.data.fdic.gov/api/financials",
        params=params
    )
    data = response.json()
    df = pd.DataFrame([item["data"] for item in data["data"]])

    # Convert numeric
    for col in [f for f in fields if f not in ['CERT', 'REPDTE']]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['CERT'] = pd.to_numeric(
        df['CERT'], errors='coerce'
    ).astype('Int64')

    return df

# ── MAIN LOOP ──────────────────────────────────────────────
all_quarters_data = []
prev_quarter_ratios = None  # For trend calculation

print("\nRunning quarter loop test...")
print("=" * 65)

for i, quarter in enumerate(test_quarters):
    start_time = time.time()
    print(f"Processing {quarter}...", end=' ')

    # Step 1: Pull raw data
    df = pull_quarter(quarter, FIELDS)

    # Step 2: Calculate ratios
    df = calculate_ratios(df)

    # Step 3: Calculate trends (QoQ)
    if prev_quarter_ratios is not None:
        # Merge previous quarter ratios
        prev_renamed = prev_quarter_ratios.rename(
            columns={col: f"{col}_PREV" for col in RATIO_COLS}
        )
        df = df.merge(prev_renamed, on='CERT', how='left')

        # Calculate trend = current - previous
        for col in RATIO_COLS:
            df[f"{col}_TREND"] = df[col] - df[f"{col}_PREV"]

        # Drop previous quarter ratio columns
        df = df.drop(
            columns=[f"{col}_PREV" for col in RATIO_COLS]
        )
    else:
        # First quarter — no trends available
        for col in RATIO_COLS:
            df[f"{col}_TREND"] = np.nan

    # Step 4: Assign default labels
    # Skip baseline quarter (2007 Q4) — only label 2008 quarters
    if i > 0:
        quarter_date = pd.Timestamp(
            f"{quarter[:4]}-{quarter[4:6]}-{quarter[6:]}"
        )
        df = assign_default_labels(
            df, quarter_date, failures_df, window_years=2
        )
        # Remove post-failure observations
        df = df[~df['POST_FAILURE']].drop(columns=['POST_FAILURE'])
    else:
        df['DEFAULT'] = np.nan

    # Step 5: Save current ratios for next quarter's trends
    prev_quarter_ratios = df[['CERT'] + RATIO_COLS].copy()

    # Step 6: Only keep training quarters (not baseline)
    if i > 0:
        all_quarters_data.append(df)

    elapsed = time.time() - start_time
    defaults = df['DEFAULT'].sum() if 'DEFAULT' in df.columns and i > 0 else 'N/A'
    trends_available = df['LEVERAGE_RATIO_TREND'].notna().sum() if i > 0 else 0

    print(f"{len(df):,} banks | "
          f"defaults: {defaults} | "
          f"trends: {trends_available:,} | "
          f"{elapsed:.1f}s")

# Combine all quarters
historical_test = pd.concat(all_quarters_data, ignore_index=True)

print("\n" + "=" * 65)
print("LOOP TEST SUMMARY")
print("=" * 65)
print(f"Quarters processed:    {len(test_quarters) - 1} "
      f"(excl. baseline)")
print(f"Total rows:            {len(historical_test):,}")
print(f"Columns:               {len(historical_test.columns)}")
print(f"Total defaults:        {historical_test['DEFAULT'].sum():,.0f}")
print(f"Overall default rate:  "
      f"{100*historical_test['DEFAULT'].mean():.2f}%")
print(f"\nColumns in dataset:")
print([c for c in historical_test.columns if 'TREND' in c])
print(f"\nMemory usage: "
      f"{historical_test.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Test quarters: ['20071231', '20080331', '20080630', '20080930', '20081231']

Running quarter loop test...
Processing 20071231... 8,632 banks | defaults: N/A | trends: 0 | 4.9s
Processing 20080331... 8,582 banks | defaults: 217 | trends: 8,525 | 4.6s
Processing 20080630... 8,538 banks | defaults: 260 | trends: 8,496 | 4.7s
Processing 20080930... 8,470 banks | defaults: 291 | trends: 8,431 | 4.7s
Processing 20081231... 8,384 banks | defaults: 304 | trends: 8,350 | 4.7s

LOOP TEST SUMMARY
Quarters processed:    4 (excl. baseline)
Total rows:            33,974
Columns:               48
Total defaults:        1,072
Overall default rate:  3.16%

Columns in dataset:
['LEVERAGE_RATIO_TREND', 'TEXAS_RATIO_TREND', 'SECURITIES_RATIO_TREND', 'UNINSURED_DEP_RATIO_TREND', 'ROA_TREND', 'LTD_RATIO_TREND', 'CRE_RATIO_TREND', 'NIM_RATIO_TREND', 'CASH_RATIO_TREND', 'BORROW_RATIO_TREND']

Memory usage: 15.6 MB


In [40]:
# ============================================================
# CELL 12: CLASS DISTRIBUTION CHECK
# ============================================================
# Purpose: Verify class imbalance exists and quantify it
#          Confirms need for class_weight='balanced'
#          in logistic regression
#
# Also validates: are the right banks being labeled default=1?
#                 Do their ratios look more stressed than
#                 default=0 banks?
# ============================================================

print("=" * 60)
print("CLASS DISTRIBUTION ANALYSIS")
print("=" * 60)

# Overall distribution
total = len(historical_test)
defaults = historical_test['DEFAULT'].sum()
healthy = total - defaults

print(f"\nOverall (4 quarters, 2008):")
print(f"  Total observations: {total:,}")
print(f"  Default = 1:        {defaults:,.0f} ({100*defaults/total:.2f}%)")
print(f"  Default = 0:        {healthy:,.0f} ({100*healthy/total:.2f}%)")
print(f"  Imbalance ratio:    1:{healthy/defaults:.0f}")
print(f"\n  → class_weight='balanced' will weight each")
print(f"    default observation {healthy/defaults:.0f}x more than healthy")

# By quarter
print(f"\nBy quarter:")
print(f"  {'Quarter':<12} {'Total':>8} {'Default=1':>10} "
      f"{'Rate':>8}")
print(f"  {'-'*42}")
for quarter in historical_test['REPDTE'].unique():
    q_data = historical_test[historical_test['REPDTE'] == quarter]
    q_total = len(q_data)
    q_def = q_data['DEFAULT'].sum()
    print(f"  {quarter:<12} {q_total:>8,} {q_def:>10,.0f} "
          f"{100*q_def/q_total:>7.2f}%")

# Key question: do default=1 banks look more stressed?
print(f"\nRatio comparison: Default=1 vs Default=0")
print(f"(Median values — do stressed banks show worse ratios?)")
print(f"\n{'Ratio':<25} {'Default=0':>12} {'Default=1':>12} "
      f"{'Direction':>10} {'✓?':>5}")
print(f"-" * 70)

for ratio in RATIO_COLS:
    median_0 = historical_test[
        historical_test['DEFAULT'] == 0][ratio].median()
    median_1 = historical_test[
        historical_test['DEFAULT'] == 1][ratio].median()
    direction = STRESS_DIRECTION[ratio]

    if direction == 'HIGH':
        correct = median_1 > median_0
        arrow = '↑ higher'
    else:
        correct = median_1 < median_0
        arrow = '↓ lower'

    status = '✅' if correct else '⚠️'
    print(f"{ratio:<25} {median_0:>12.4f} {median_1:>12.4f} "
          f"{arrow:>10} {status:>5}")

print(f"\nConclusion:")
print(f"  If all ratios show ✅ — default banks are measurably")
print(f"  more stressed than healthy banks across all dimensions")
print(f"  This confirms ratios have predictive power for default")

CLASS DISTRIBUTION ANALYSIS

Overall (4 quarters, 2008):
  Total observations: 33,974
  Default = 1:        1,072 (3.16%)
  Default = 0:        32,902 (96.84%)
  Imbalance ratio:    1:31

  → class_weight='balanced' will weight each
    default observation 31x more than healthy

By quarter:
  Quarter         Total  Default=1     Rate
  ------------------------------------------
  20080331        8,582        217    2.53%
  20080630        8,538        260    3.05%
  20080930        8,470        291    3.44%
  20081231        8,384        304    3.63%

Ratio comparison: Default=1 vs Default=0
(Median values — do stressed banks show worse ratios?)

Ratio                        Default=0    Default=1  Direction    ✓?
----------------------------------------------------------------------
LEVERAGE_RATIO                  0.0951       0.0763    ↓ lower     ✅
TEXAS_RATIO                     0.0743       0.5945   ↑ higher     ✅
SECURITIES_RATIO                0.1682       0.0913   ↑ higher    ⚠